# 미니 KORMo-VL — 1B 백본에 눈 달아주기 (Colab A100/L4)

`01` 노트북으로 사전학습한 **KORMo 1B(`output/final/`)를 LLM 백본**으로 쓰고,
[KORMo-VL](https://hf.co/KORMo-VL/KORMo-VL)과 같은 **LLaVA 방식**으로 비전 능력을 붙이는 실습입니다.

```
이미지 → SigLIP 비전 인코더(동결) → 프로젝터 MLP(학습!) → [이미지 토큰 196개] + [텍스트 토큰]
                                                              ↓
                                                    KORMo 1B (동결) → 다음 토큰 예측
```

- 학습 대상은 **프로젝터(2층 MLP, ~7M 파라미터)뿐** — 비전 인코더와 LLM은 동결
- 본가 KORMo-VL(10B+siglip-so400m, 3.4M쌍, 3단계)의 **1단계(정렬)를 축소 재현**하되,
  캡션 대신 한국어 VQA 데이터(KoLLaVA)로 단순화해 정렬+응답을 한 번에 학습합니다
- **전제**: Drive `kormo-1B-PT/output/final/`에 1번 노트북의 학습 완료 모델 존재

예상 소요: 데이터 ~1GB 다운로드 + 학습 20~40분 (A100 기준, L4는 배치를 절반으로)

In [ ]:
# GPU 확인 — A100 권장, L4면 BATCH_SIZE를 4로 낮추세요
!nvidia-smi -L

## 0. 환경 준비

1번 노트북과 같은 스택에 SigLIP(비전 인코더)용 처리만 추가됩니다.

In [ ]:
import os
# HF 다운로드 경로 — 토큰 유무에 따라 분기 (huggingface_hub import 전에 결정해야 함):
# - HF_TOKEN 있음: Xet 네이티브 프로토콜 사용 (브리지 CDN 403 우회)
# - HF_TOKEN 없음: Xet 비활성 → 일반 HTTP 폴백
FORCE_DISABLE_XET = False   # 다운로드가 계속 실패하면 True로 바꿔 일반 HTTP로 강제 폴백

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("HF_TOKEN 로드됨 — Xet 네이티브 경로 사용")
except Exception:
    os.environ['HF_HUB_DISABLE_XET'] = '1'
    print("HF_TOKEN 없음 — Xet 비활성 (브리지 CDN 폴백)")

if FORCE_DISABLE_XET:
    os.environ['HF_HUB_DISABLE_XET'] = '1'
    print("FORCE_DISABLE_XET — Xet 비활성 (일반 HTTP 폴백)")

!git clone -q https://github.com/MLP-Lab/KORMo-tutorial.git /content/KORMo-tutorial
%pip install -q "transformers==4.57.1" "datasets>=4.1.1" hf_xet

import sys, json
sys.path.append('/content/KORMo-tutorial/src')

### Google Drive — 백본 로드 + 프로젝터 저장

| 경로 | 용도 |
|---|---|
| `kormo-1B-PT/output/final/` | 1번 노트북이 만든 LLM 백본 (읽기 전용 — 덮어쓰지 않음) |
| `kormo-1B-PT/vl/` | 이 노트북의 산출물 (프로젝터 체크포인트·final) |

`vl/final/projector.pt`가 이미 있으면 이어서 학습(continue)합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/kormo-1B-PT'
LM_DIR = os.path.join(BASE_DIR, 'output', 'final')
if not os.path.isdir(LM_DIR) and os.path.isdir(os.path.join(BASE_DIR, 'final')):
    LM_DIR = os.path.join(BASE_DIR, 'final')   # 구버전 레이아웃
assert os.path.isdir(LM_DIR), f"{LM_DIR} 가 없습니다 — 1번 노트북(사전학습)을 먼저 완주하세요."

VL_DIR = os.path.join(BASE_DIR, 'vl')
os.makedirs(VL_DIR, exist_ok=True)
PROJECTOR_CKPT = os.path.join(VL_DIR, 'projector_last.pt')   # 학습 중 주기 저장
PROJECTOR_FINAL = os.path.join(VL_DIR, 'final', 'projector.pt')

print("LLM 백본:", LM_DIR)
print("VL 산출물:", VL_DIR)

## 1. 데이터 — KoLLaVA (한국어 시각 질의응답)

[mosshoon/KoLLaVA-Instruct-313k](https://hf.co/datasets/mosshoon/KoLLaVA-Instruct-313k):
LLaVA-Instruct-150k를 한국어로 번역한 데이터의 이미지 내장(parquet) 버전.
총 33개 샤드(16GB)인데 **샤드 단위로 골라 받을 수 있어** 기본 2개(~1GB, 약 19k쌍)만 사용합니다.

- 본가 LLaVA 1단계는 캡션 쌍(CC3M)으로 프로젝터만 정렬하지만, 튜토리얼에서는
  VQA 데이터 하나로 정렬을 겸합니다 — 손실은 **답변 토큰에만** 걸립니다
- `SMOKE = True`면 1.5k 미니 데이터셋(79MB)으로 파이프라인만 빠르게 검증

In [ ]:
import time, datasets

SMOKE = False        # True: 1.5k 미니 데이터로 동작 검증
NUM_SHARDS = 2       # 313k 중 사용할 샤드 수 (샤드당 ~492MB, ~9.5k쌍)

def retry_wait(e, attempt, max_attempts=5):
    """에러 내용을 그대로 출력하고 상태 코드에 맞게 대기/중단 결정."""
    code = getattr(getattr(e, 'response', None), 'status_code', None)
    print(f"[{attempt}/{max_attempts}] 실패: {type(e).__name__}(HTTP {code}) — {str(e)[:300]}")
    if code == 401:
        raise RuntimeError("401 인증 실패 — Colab 보안 비밀 HF_TOKEN을 확인하세요") from e
    wait = 90 * attempt if code == 429 else 30
    print(f"  → {wait}초 후 재시도")
    time.sleep(wait)

raw = None
for attempt in range(1, 6):
    try:
        if SMOKE:
            raw = datasets.load_dataset('mosshoon/KoLLaVA-Instruct-1.5k', split='train')
        else:
            shards = [f'data/train-{i:05d}-of-00033.parquet' for i in range(NUM_SHARDS)]
            raw = datasets.load_dataset('mosshoon/KoLLaVA-Instruct-313k',
                                        data_files=shards, split='train')
        break
    except Exception as e:
        retry_wait(e, attempt)
assert raw is not None, "5회 재시도 실패 — 에러 메시지 확인 (403이면 FORCE_DISABLE_XET=True로 재시도)"

split = raw.train_test_split(test_size=64, seed=42)
train_ds, val_ds = split['train'], split['test']
print(f"학습 {len(train_ds)}쌍 | 검증 {len(val_ds)}쌍")
print("예시 질문:", train_ds[0]['questions'].replace('<image>', '').strip()[:80])
print("예시 답변:", train_ds[0]['answers'][:80])

## 2. 모델 조립 — SigLIP(동결) + 프로젝터(학습) + KORMo 1B(동결)

- **비전 인코더**: `google/siglip-base-patch16-224` (86M, 224px → 14×14 = **196 패치 토큰**, hidden 768).
  본가는 so400m-384(729토큰)를 쓰지만 튜토리얼 예산에 맞춰 축소
- **프로젝터**: LLaVA-1.5와 같은 2층 MLP — `768 → 2048 → 2048` (~7M). 유일한 학습 대상이라 **fp32**로 두고,
  출력만 bf16으로 캐스팅해 LLM에 전달합니다 (작은 모듈은 fp32가 안정적)
- **LLM**: 1번 노트북의 `final/`을 bf16 + sdpa로 로드, 완전 동결

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, SiglipVisionModel, SiglipImageProcessor
from kormo.model._modeling_kormo import KORMoForCausalLM

VISION_NAME = 'google/siglip-base-patch16-224'

vision = SiglipVisionModel.from_pretrained(VISION_NAME, dtype=torch.bfloat16).to('cuda').eval()
processor = SiglipImageProcessor.from_pretrained(VISION_NAME)
N_IMG_TOKENS = (vision.config.image_size // vision.config.patch_size) ** 2

lm = KORMoForCausalLM.from_pretrained(
    LM_DIR, dtype=torch.bfloat16, _attn_implementation='sdpa').to('cuda').eval()
embed_tokens = lm.get_input_embeddings()

tokenizer = AutoTokenizer.from_pretrained(LM_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

projector = nn.Sequential(
    nn.Linear(vision.config.hidden_size, lm.config.hidden_size),
    nn.GELU(),
    nn.Linear(lm.config.hidden_size, lm.config.hidden_size),
).to('cuda')   # fp32

if os.path.exists(PROJECTOR_FINAL):
    projector.load_state_dict(torch.load(PROJECTOR_FINAL, map_location='cuda'))
    print("기존 프로젝터 로드 (continue):", PROJECTOR_FINAL)

for p in vision.parameters():
    p.requires_grad_(False)
for p in lm.parameters():
    p.requires_grad_(False)

n_total = sum(p.numel() for p in vision.parameters()) + sum(p.numel() for p in lm.parameters())
n_train = sum(p.numel() for p in projector.parameters())
print(f"동결 {n_total/1e9:.2f}B | 학습 {n_train/1e6:.1f}M (전체의 {100*n_train/(n_total+n_train):.3f}%)")
print(f"이미지 1장 = {N_IMG_TOKENS} 토큰")

### Collator + 공용 forward

시퀀스 구성: `[이미지 196토큰] [BOS 질문...\n] [답변... EOS]`
- 손실(label)은 **답변 구간에만** — 이미지·질문 구간은 -100
- 패딩 위치도 -100 + attention 0

In [ ]:
from dataclasses import dataclass
from torch.nn.utils.rnn import pad_sequence

MAX_TXT = 320   # 질문+답변 최대 토큰 (KoLLaVA 응답이 길어 넉넉히)

def collate(examples):
    images = [ex['images'].convert('RGB') for ex in examples]
    pixel_values = processor(images=images, return_tensors='pt')['pixel_values']
    ids_list, labels_list = [], []
    for ex in examples:
        q = ex['questions'].replace('<image>', '').strip()
        prompt_ids = tokenizer.encode(q + '\n')                                   # BOS 포함
        answer_ids = tokenizer.encode(ex['answers'], add_special_tokens=False) + [tokenizer.eos_token_id]
        ids = (prompt_ids + answer_ids)[:MAX_TXT]
        labs = ([-100] * len(prompt_ids) + answer_ids)[:MAX_TXT]
        ids_list.append(torch.tensor(ids))
        labels_list.append(torch.tensor(labs))
    input_ids = pad_sequence(ids_list, batch_first=True, padding_value=tokenizer.pad_token_id)
    labels = pad_sequence(labels_list, batch_first=True, padding_value=-100)
    text_mask = (input_ids != tokenizer.pad_token_id).long()
    return dict(pixel_values=pixel_values, input_ids=input_ids, labels=labels, text_mask=text_mask)

def vl_forward(batch):
    """[이미지 임베딩]+[텍스트 임베딩]을 이어 LLM에 통과 — 학습·평가·생성이 공유."""
    pixel_values = batch['pixel_values'].to('cuda', torch.bfloat16)
    input_ids = batch['input_ids'].to('cuda')
    with torch.no_grad():
        vis = vision(pixel_values=pixel_values).last_hidden_state       # [B, 196, 768]
    img_embeds = projector(vis.float()).to(torch.bfloat16)              # 프로젝터만 grad
    txt_embeds = embed_tokens(input_ids)
    inputs_embeds = torch.cat([img_embeds, txt_embeds], dim=1)
    B = input_ids.shape[0]
    img_pad = torch.full((B, N_IMG_TOKENS), -100, device='cuda', dtype=torch.long)
    labels = torch.cat([img_pad, batch['labels'].to('cuda')], dim=1)
    attention_mask = torch.cat(
        [torch.ones(B, N_IMG_TOKENS, device='cuda', dtype=torch.long),
         batch['text_mask'].to('cuda')], dim=1)
    return lm(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels)

## 3. 학습 — 프로젝터 정렬

- **LR 1e-3** — LLaVA 1단계 관례. 무작위 초기화된 작은 MLP만 학습하므로 LLM 학습률(5e-4)보다 커도 안전
- 초기 loss는 ~11(무작위) → 이미지-텍스트 정렬이 되면서 2~4 대로 내려가는 것이 정상
- 프로젝터(~28MB)를 300스텝마다 Drive에 저장 — 세션이 끊겨도 이어서 학습 가능

In [ ]:
import math
from torch.utils.data import DataLoader

BATCH_SIZE = 8      # L4면 4로
EPOCHS = 1
LR = 1e-3
LOG_EVERY, EVAL_EVERY, SAVE_EVERY = 20, 200, 300
MAX_STEPS = 30 if SMOKE else -1

loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    collate_fn=collate, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, collate_fn=collate)
total_steps = MAX_STEPS if MAX_STEPS > 0 else len(loader) * EPOCHS
warmup = max(10, int(total_steps * 0.03))

opt = torch.optim.AdamW(projector.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: s / warmup if s < warmup else
    0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total_steps - warmup))))

@torch.no_grad()
def eval_loss():
    projector.eval()
    tot, n = 0.0, 0
    for b in val_loader:
        tot += vl_forward(b).loss.item(); n += 1
    projector.train()
    return tot / n

step, train_log, eval_log = 0, [], []
projector.train()
for epoch in range(EPOCHS):
    for batch in loader:
        out = vl_forward(batch)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(projector.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        step += 1
        if step % LOG_EVERY == 0:
            train_log.append((step, out.loss.item()))
            print(f"step {step}/{total_steps} | loss {out.loss.item():.3f} | lr {sched.get_last_lr()[0]:.2e}")
        if step % EVAL_EVERY == 0:
            ev = eval_loss(); eval_log.append((step, ev))
            print(f"  └ eval loss {ev:.3f}")
        if step % SAVE_EVERY == 0:
            torch.save(projector.state_dict(), PROJECTOR_CKPT)
        if MAX_STEPS > 0 and step >= MAX_STEPS:
            break
    else:
        continue
    break

ev = eval_loss(); eval_log.append((step, ev))
print(f"학습 종료 — 최종 eval loss {ev:.3f}")

### 손실 곡선

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
if train_log:
    ax.plot(*zip(*train_log), label='train', alpha=0.7)
if eval_log:
    ax.plot(*zip(*eval_log), marker='o', label='eval')
ax.set_xlabel('step'); ax.set_ylabel('loss')
ax.set_title('Projector alignment — loss')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 4. 생성 데모 — 모델이 이미지를 "보고" 말하는지

수동 greedy 루프로 생성합니다 (KV 캐시 없이 매 스텝 전체 forward — 느리지만 로직이 투명).
프로젝터만 잠깐 학습한 상태라 **어색하지만 이미지와 관련 있는 한국어**가 나오면 성공입니다.
비교를 위해 프로젝터를 통과하지 않은 **무작위 이미지 임베딩**으로도 생성해봅니다.

In [ ]:
@torch.no_grad()
def generate(pil_image, prompt="이 이미지를 설명해주세요.", max_new_tokens=64):
    projector.eval()
    pv = processor(images=pil_image.convert('RGB'), return_tensors='pt')['pixel_values']
    vis = vision(pixel_values=pv.to('cuda', torch.bfloat16)).last_hidden_state
    img_embeds = projector(vis.float()).to(torch.bfloat16)
    ids = torch.tensor([tokenizer.encode(prompt + '\n')], device='cuda')
    cur = torch.cat([img_embeds, embed_tokens(ids)], dim=1)
    out_tokens = []
    for _ in range(max_new_tokens):
        mask = torch.ones(cur.shape[:2], device='cuda', dtype=torch.long)
        logits = lm(inputs_embeds=cur, attention_mask=mask).logits
        nxt = logits[:, -1].argmax(-1)
        if nxt.item() == tokenizer.eos_token_id:
            break
        out_tokens.append(nxt.item())
        cur = torch.cat([cur, embed_tokens(nxt[None])], dim=1)
    return tokenizer.decode(out_tokens)

for i in range(3):
    ex = val_ds[i]
    plt.figure(figsize=(3.5, 3.5)); plt.imshow(ex['images']); plt.axis('off'); plt.show()
    print("생성:", generate(ex['images']))
    print("정답 예시:", ex['answers'][:120], "\n")

## 5. 최종 저장 + 다음 단계

프로젝터와 조립 정보를 `vl/final/`에 저장합니다. LLM 백본(`output/final/`)은 건드리지 않았으므로
1·2번 노트북의 학습/평가 사이클과 독립적으로 공존합니다.

**다음 단계 아이디어**
1. `NUM_SHARDS`를 늘려 정렬 품질 확인 (손실·생성 비교)
2. **2단계: LLM 언프리징** — 프로젝터+LLM을 낮은 LR(2e-5)로 함께 튜닝 (본가 KORMo-VL의 instruction tuning에 대응, LoRA 권장)
3. 비전 인코더를 so400m-384로 교체해 해상도 효과 확인
4. `02` 평가 노트북처럼 생성 결과를 Drive에 누적해 정렬 품질 추이 기록

In [ ]:
os.makedirs(os.path.dirname(PROJECTOR_FINAL), exist_ok=True)
torch.save(projector.state_dict(), PROJECTOR_FINAL)
with open(os.path.join(VL_DIR, 'final', 'config.json'), 'w') as f:
    json.dump({'vision': VISION_NAME, 'lm_dir': LM_DIR, 'n_img_tokens': N_IMG_TOKENS,
               'projector': 'mlp2x_gelu', 'trained_pairs': len(train_ds), 'epochs': EPOCHS}, f, indent=2)
print("저장 완료:", PROJECTOR_FINAL)